# Linear Regression with Neural Networks (v3)
**Student:** Daniel Sozoranga
**Course:** Machine Learning
**Assignment:** Recovery (extra credit)

## Objective
Predict the **annual medical insurance cost** from personal features (age, BMI, children, smoker, region, sex) using **neural networks** with TensorFlow/Keras.

Three **models of the same problem** are trained to compare learning states:
1. **Well-trained** — correct bias/variance balance, with Dropout.
2. **Under-trained (underfitting)** — the model cannot learn the signal.
3. **Over-trained (overfitting)** — the model memorizes the train set and fails on test.

## Short theoretical framework
**Regression** predicts a continuous numeric value. A neural network performs regression when its output layer has **a single neuron with linear activation**. Without hidden layers it reduces to classical linear regression; with non-linear hidden layers it can learn non-linear relationships.

### Cake analogy
- **Features (ingredients):** age, BMI, smoker, etc.
- **Parameters (recipe):** the weights the network adjusts at every epoch.
- **Epochs:** each full pass over the data.
- **Underfitting:** recipe too simple, cake raw.
- **Overfitting:** we memorized THAT particular cake, cannot bake another one.
- **Well-trained:** a generalizable recipe — the cake turns out well every time.


## 1. Imports


**Libraries.** `numpy` / `pandas` for numerical and tabular data handling, `matplotlib` for plots, `tensorflow/keras` to build and train the neural networks, and `sklearn` utilities for train/test splitting, scaling, metrics, cross-validation and interpretability. A seed (42) is fixed so that results are reproducible across runs.


In [ ]:
# Suppress TensorFlow retracing warnings and info logs for cleaner notebook output.
# These warnings do not indicate errors; they just clutter the output during K-fold CV.
import warnings, os
warnings.filterwarnings('ignore')                  # Silence Python-level warnings (deprecations, etc.)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'           # 0=all, 1=no INFO, 2=no WARNING, 3=only ERROR

# Core scientific stack
import numpy as np                                 # Numerical arrays and vectorized math
import pandas as pd                                # Tabular data handling (DataFrame, read_csv, etc.)
import matplotlib.pyplot as plt                    # Plotting library for loss curves, residuals, etc.

# Deep learning framework
import tensorflow as tf                            # TensorFlow backend
from tensorflow import keras                       # High-level Keras API for building models
from tensorflow.keras import layers, regularizers  # Layer primitives (Dense, Dropout, ...) and L1/L2 regularizers
tf.get_logger().setLevel('ERROR')                  # Silence the TF Python-side logger as well

# Classic ML utilities from scikit-learn
from sklearn.model_selection import train_test_split, KFold, learning_curve, validation_curve
# train_test_split: random train/test partition
# KFold: K-fold cross-validation splitter
# learning_curve / validation_curve: diagnostic curves for bias/variance

from sklearn.preprocessing import StandardScaler   # Mean-0 / std-1 feature scaler
from sklearn.linear_model import LinearRegression  # Classical OLS linear regression (baseline)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# MSE: mean squared error; MAE: mean absolute error; R2: coefficient of determination

from sklearn.inspection import permutation_importance   # Model-agnostic feature importance
from sklearn.base import BaseEstimator, RegressorMixin  # Base classes to wrap Keras as sklearn regressor

# Reproducibility: fix seeds so that results are stable across runs.
# NOTE: TF is not 100% deterministic on GPU, but with seeds + CPU it is very close.
np.random.seed(42); tf.random.set_seed(42)


## 2. Dataset loading


**Dataset.** The well-known US medical insurance dataset is loaded (1,338 records). The target `charges` is numeric and continuous → **regression** problem. Features: age, sex, BMI, number of children, smoker/non-smoker and region.


In [ ]:
# Public insurance dataset from the R "Machine Learning with R" book repository.
# 1,338 rows, 7 columns. Target `charges` is a continuous variable (USD) -> regression problem.
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

df = pd.read_csv(URL)                              # Download and parse the CSV into a DataFrame
print("Dimensions:", df.shape)                     # Expected: (1338, 7)
df.head()                                          # Preview the first 5 rows to confirm columns


**Descriptive statistics.** Useful to detect very different scales across variables (age 18–64 vs charges 1k–64k). This justifies standardizing the features and transforming the target to correct its skewness.


In [ ]:
# Descriptive statistics per numerical column.
# Useful to check the scale disparity: `age` spans 18-64 while `charges` spans ~1k-64k USD.
# This scale gap justifies standardizing features and transforming the target.
df.describe()


## 3. Preprocessing
One-hot encoding of categoricals, 80/20 split and standardization. `log1p` applied to the target to correct its skewness.


**Preprocessing.** Categorical variables are encoded with *one-hot encoding* (`drop_first=True` avoids perfect multicollinearity). The data is split 80% train / 20% test randomly with a fixed seed. `StandardScaler` (mean 0, standard deviation 1) is applied to the features — the scaler is fit **only** on the training set to avoid *data leakage*. `log1p` is applied to the target because `charges` has strong positive skew: the logarithmic transform makes the distribution more symmetric and stabilizes the network's training.


In [ ]:
# --- One-hot encode the categorical columns ---
# `drop_first=True` drops one dummy level per category to avoid perfect multicollinearity
# (the dummy-variable trap). `astype(float)` ensures the booleans become 0.0/1.0 numerics.
df_enc = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True).astype(float)

# --- Split features and target ---
X = df_enc.drop('charges', axis=1).values          # Feature matrix as numpy array (shape: N x D)
y = df_enc['charges'].values                       # Target vector in original USD scale
feat_names = df_enc.drop('charges', axis=1).columns.tolist()  # Keep feature names for later (importance plot)

# --- Log-transform the target ---
# `charges` is right-skewed (long tail of high medical costs). log1p(y) = log(1 + y)
# symmetrizes the distribution and stabilizes gradients during NN training.
# We keep both `y` (original USD) and `y_log` because the OVER model will train on raw USD later.
y_log = np.log1p(y)

# --- Train/test split (80/20) ---
# random_state=42 makes the split reproducible. We carry both log and raw targets so every
# model can choose the scale it trains on, while evaluation is always against raw USD `y_te`.
X_tr, X_te, y_tr_log, y_te_log, y_tr, y_te = train_test_split(
    X, y_log, y, test_size=0.2, random_state=42)

# --- Standardize features ---
# IMPORTANT: fit the scaler ONLY on training data, then reuse it to transform test data.
# Fitting on the full dataset would leak test-set statistics into training (data leakage).
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)                # Learn mean/std from train, then transform
X_te_s = scaler.transform(X_te)                    # Apply the same transformation to test

print("Train:", X_tr_s.shape, " Test:", X_te_s.shape)


## 4. Loss function and metrics

**Important — this is regression, not classification.** The output layer of all three networks has **1 neuron with linear activation** (`Dense(1)` without an `activation=` argument), which defines the problem as **regression**: the network predicts a continuous numeric value (cost in USD). A classification network would use `sigmoid` (binary) or `softmax` (multi-class) with a `binary_crossentropy` / `categorical_crossentropy` loss. Here we use `mse` because `charges` is continuous.

- **MSE (loss):** differentiable, penalizes large errors, standard choice for regression.
- **MAE:** mean absolute error in USD, robust to outliers.
- **RMSE:** √MSE, interpretable in units of the target.
- **R²:** fraction of variance explained (1.0 perfect, 0 equals predicting the mean, negative is worse than the mean).
- **Robust alternative: Huber Loss** (quadratic near 0, linear far from 0). Useful if extreme outliers were present.


## 5. Baseline model: Linear Regression (reported with and without log1p)
A fair comparison to prove that the NN adds value over the baseline.


**Baseline model: Linear Regression.** A classical linear model is trained as a *baseline*. A neural network is only justified if it clearly beats this starting point; otherwise it means there are no relevant non-linear relationships to capture.


In [ ]:
# --- Baseline 1: Linear Regression on the ORIGINAL target scale ---
# This is the honest baseline the NN must beat to justify its existence.
lin_raw = LinearRegression().fit(X_tr_s, y_tr)     # Fit OLS on standardized features, raw USD target
pred_lin_raw = lin_raw.predict(X_te_s)             # Predict test set
print(f"LR on raw y       -> MAE={mean_absolute_error(y_te, pred_lin_raw):,.2f} | "
      f"RMSE={np.sqrt(mean_squared_error(y_te, pred_lin_raw)):,.2f} | "
      f"R2={r2_score(y_te, pred_lin_raw):.4f}")

# --- Baseline 2: Linear Regression on LOG1P(target) ---
# Same model but trained on log-transformed target. Predictions are converted back to USD
# via expm1(pred) = exp(pred) - 1, which is the inverse of log1p.
# Reported for completeness to show both baselines under identical preprocessing.
lin_log = LinearRegression().fit(X_tr_s, y_tr_log) # Fit OLS on log-scale target
pred_lin_log = np.expm1(lin_log.predict(X_te_s))   # Predict log-scale, invert back to USD
print(f"LR on log1p(y)    -> MAE={mean_absolute_error(y_te, pred_lin_log):,.2f} | "
      f"RMSE={np.sqrt(mean_squared_error(y_te, pred_lin_log)):,.2f} | "
      f"R2={r2_score(y_te, pred_lin_log):.4f}")

# We keep the raw-scale LR as THE baseline for later comparison (fairer apples-to-apples
# against the NN that also evaluates in USD).
pred_lin = pred_lin_raw


## 6. Definition and training of the neural network (well-trained)

The **well-trained** model is defined and trained first: moderate architecture (64→32 neurons) with `Dropout(0.1)` for regularization and `EarlyStopping` to stop training when validation loss stops improving. Trained on `log1p(y)` to account for the positive skew of `charges`.

**Bias correction (smearing).** When `expm1` is applied to invert `log1p` back to USD, predictions end up systematically below the true values (the logarithm is concave, so `E[exp(X)] ≠ exp(E[X])`). A constant offset is computed on the training set and added to test predictions, centering the residuals near zero.


In [ ]:
# ---------- WELL-TRAINED MODEL: well-regularized, balanced capacity ----------
def build_good(n):
    """
    Moderate architecture: 64 -> 32 hidden units with ReLU, light Dropout for regularization.
    Final layer is Dense(1) with NO activation -> linear output -> REGRESSION.
    (A classifier would use sigmoid/softmax here; we want a continuous USD prediction.)
    """
    m = keras.Sequential([
        layers.Input(shape=(n,)),                  # Input layer: n features
        layers.Dense(64, activation='relu'),       # First hidden layer, 64 units, ReLU non-linearity
        layers.Dropout(0.1),                       # Randomly zero 10% of activations during training (regularization)
        layers.Dense(32, activation='relu'),       # Second hidden layer, 32 units
        layers.Dropout(0.1),                       # Second dropout block
        layers.Dense(1)                            # OUTPUT: 1 neuron, linear activation -> scalar regression
    ])
    # Adam optimizer with lr=1e-3 (Keras default, stable). Loss = MSE (standard for regression).
    # MAE tracked as human-interpretable secondary metric.
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

# Re-seed all RNGs right before training for stronger reproducibility.
tf.keras.utils.set_random_seed(42)

# EarlyStopping callback: stop training when val_loss has not improved for 50 consecutive epochs,
# and roll back to the best weights seen.
early_stop = keras.callbacks.EarlyStopping(patience=50, restore_best_weights=True, monitor='val_loss')

# ---------- Train GOOD model ----------
model_good = build_good(X_tr_s.shape[1])           # Instantiate the architecture
hist_good = model_good.fit(
    X_tr_s, y_tr_log,                              # Train on log-transformed target
    validation_split=0.2,                          # Hold out 20% of train for validation monitoring
    epochs=500,                                    # Upper cap; EarlyStopping will cut much earlier
    batch_size=32,                                 # Standard batch size
    callbacks=[early_stop],                        # Enable EarlyStopping
    verbose=0)                                     # Silent training (we'll plot the history)

# ---------- Smearing / bias correction ----------
# When we invert log1p via expm1, predictions are systematically shifted downward because
# log is concave: E[exp(X)] != exp(E[X]) (Jensen's inequality). We estimate a constant
# correction on TRAIN (never on test) equal to the mean residual, and add it to future
# USD predictions to center the residuals around zero.
pred_tr_raw_good = np.expm1(model_good.predict(X_tr_s, verbose=0).ravel())  # Train predictions in USD
bias_corr = (y_tr - pred_tr_raw_good).mean()                                # Mean residual -> constant offset
print(f"Bias correction applied to the GOOD model: {bias_corr:,.2f} USD")


## 7. Loss curve (well-trained)

Plot of `train loss` vs `val loss` across epochs for the well-trained model. Curves should be close together and low, indicating good generalization.


In [ ]:
# Loss curves for the well-trained model
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(hist_good.history['loss'],     label='train')
ax.plot(hist_good.history['val_loss'], label='val')
ax.set_title('Well-trained NN')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE (log-scale target)')
ax.legend()
plt.tight_layout(); plt.show()


## 8. Evaluation (well-trained vs baseline)

Comparison between the well-trained NN and the Linear Regression baseline. MAE is in USD, RMSE penalizes large errors more, and R² is the fraction of variance explained.


In [ ]:
# Helper: evaluate a model trained on LOG-scale target (applies expm1 + bias correction)
def evaluar_log(model, nombre, corr=0.0):
    pred = np.expm1(model.predict(X_te_s, verbose=0).ravel()) + corr
    return {
        'Model': nombre,
        'MAE':    mean_absolute_error(y_te, pred),
        'RMSE':   np.sqrt(mean_squared_error(y_te, pred)),
        'R2':     r2_score(y_te, pred)
    }

# Partial comparison table: LR baseline + NN well-trained only for now.
resultados = pd.DataFrame([
    {'Model': 'LinearRegression (baseline)',
     'MAE':  mean_absolute_error(y_te, pred_lin),
     'RMSE': np.sqrt(mean_squared_error(y_te, pred_lin)),
     'R2':   r2_score(y_te, pred_lin)},
    evaluar_log(model_good, 'NN well-trained', corr=bias_corr),
])
resultados
